In [1]:
import glob, os, shutil, subprocess, sys, json, math, time, zipfile
from pathlib import Path

RUN_HH = True   # HH-RLHF DPO
RUN_UF = True   # UltraFeedback DPO

WORK_DIR = Path("/kaggle/working/PA2")
os.environ["PYTHONUNBUFFERED"] = "1"

def run(cmd):
    """Run a shell command, stream its output, raise on failure."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


In [2]:
REQUIRED = [
    "pyproject.toml", "submission_adapter.py", "pa2/sft.py", "pa2/dpo.py", "pa2/data.py",
    "scripts/train_dpo.py", "scripts/generate.py",
    "outputs/sft/best/config.json", "outputs/sft/best/model.safetensors",
    "eval_sets/hh_rlhf_eval_300_seed42.json",
]

def find_root():
    hits = glob.glob("/kaggle/input/**/scripts/train_dpo.py", recursive=True)
    roots = [Path(h).parent.parent for h in hits]
    good = [r for r in roots if (r / "outputs/sft/best/config.json").exists()]
    return good[0] if good else None

if not (WORK_DIR / "pyproject.toml").exists():
    root = find_root()
    if root is None:
        raise FileNotFoundError("No PA2 folder with scripts/train_dpo.py AND outputs/sft/best found under /kaggle/input. "
                                "Attach your consolidated dataset.")
    print("Using:", root)
    shutil.copytree(root, WORK_DIR, dirs_exist_ok=True)

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
for folder in ("outputs/gen", "outputs/eval", "outputs/dpo_hh", "outputs/dpo_uf", "eval_sets"):
    Path(folder).mkdir(parents=True, exist_ok=True)

missing = [p for p in REQUIRED if not Path(p).exists()]
for p in REQUIRED + ["outputs/gen/base_alpaca.json", "eval_sets/alpaca_eval_300_seed42.json"]:
    print(f"{p:45s} {'OK' if Path(p).exists() else 'MISSING'}")
assert not missing, f"Missing required files: {missing}"

SFT = str(WORK_DIR / "outputs/sft/best")
print("\nSFT checkpoint:", SFT)


Using: /kaggle/input/datasets/areeshayaseen/cs5326-pa2-dpo/PA2
pyproject.toml                                OK
submission_adapter.py                         OK
pa2/sft.py                                    OK
pa2/dpo.py                                    OK
pa2/data.py                                   OK
scripts/train_dpo.py                          OK
scripts/generate.py                           OK
outputs/sft/best/config.json                  OK
outputs/sft/best/model.safetensors            OK
eval_sets/hh_rlhf_eval_300_seed42.json        OK
outputs/gen/base_alpaca.json                  OK
eval_sets/alpaca_eval_300_seed42.json         OK

SFT checkpoint: /kaggle/working/PA2/outputs/sft/best


**Install Dependencies**

In [3]:
%cd /kaggle/working/PA2
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps
!nvidia-smi -L

import torch, transformers, datasets
print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| datasets:", datasets.__version__)
print("GPU count:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "DPO needs 2 GPUs (policy on cuda:0, frozen reference on cuda:1). Select GPU T4 x2."


/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 10.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment-starter (pyp

In [4]:
%cd /kaggle/working/PA2
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

import torch
from submission_adapter import compute_dpo_loss_from_logps as dpo_from_logps
x = torch.tensor([-10.0, -20.0])
loss = dpo_from_logps(x, x - 3, x, x - 3, 0.1)
print("DPO loss when policy == reference:", float(loss), "| ln2 =", math.log(2))
assert abs(float(loss) - math.log(2)) < 1e-5


/kaggle/working
.......                                                                  [100%]
7 passed in 2.26s
DPO loss when policy == reference: 0.6931471824645996 | ln2 = 0.6931471805599453


**DPO on HH-RLHF**

In [5]:
%cd /kaggle/working/PA2
out = Path("outputs/dpo_hh")
if not RUN_HH:
    print("RUN_HH is False -> skipping")
elif (out / "final" / "config.json").exists():
    print("HH DPO already finished -> skipping")
else:
    t0 = time.time()
    run(f"python scripts/train_dpo.py --dataset hh --sft_path {SFT} --out_dir outputs/dpo_hh")
    print(f"HH DPO finished in {(time.time()-t0)/3600:.2f} h")

/kaggle/working
$ python scripts/train_dpo.py --dataset hh --sft_path /kaggle/working/PA2/outputs/sft/best --out_dir outputs/dpo_hh
The tokenizer you are loading from '/kaggle/working/PA2/outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.

Generating train split: 100%|██████████| 160800/160800 [00:02<00:00, 64158.64 examples/s]

Generating test split: 100%|██████████| 8552/8552 [00:00<00:00, 66401.00 examples/s]
15000 train / 1000 val pairs after prompt-length filter
The tokenizer you are loading from '/kaggle/working/PA2/outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `f

**DPO on UltraFeedback**

In [6]:
%cd /kaggle/working/PA2
out = Path("outputs/dpo_uf")
if not RUN_UF:
    print("RUN_UF is False -> skipping")
elif (out / "final" / "config.json").exists():
    print("UltraFeedback DPO already finished -> skipping")
else:
    t0 = time.time()
    run(f"python scripts/train_dpo.py --dataset ultrafeedback --sft_path {SFT} --out_dir outputs/dpo_uf")
    print(f"UltraFeedback DPO finished in {(time.time()-t0)/3600:.2f} h")

/kaggle/working
$ python scripts/train_dpo.py --dataset ultrafeedback --sft_path /kaggle/working/PA2/outputs/sft/best --out_dir outputs/dpo_uf
The tokenizer you are loading from '/kaggle/working/PA2/outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.

Generating train_prefs split: 100%|██████████| 61135/61135 [00:01<00:00, 49615.69 examples/s]

Generating train_sft split: 100%|██████████| 61135/61135 [00:00<00:00, 70425.20 examples/s]

Generating test_prefs split: 100%|██████████| 2000/2000 [00:00<00:00, 66993.63 examples/s]

Generating test_sft split: 100%|██████████| 1000/1000 [00:00<00:00, 64705.94 examples/s]

Generating train_gen split: 100%|██████████| 61135/61135 [00:00<00:00, 90178.75 examples/s]

Generating test_gen split: 100%|██████

**Verification and Summerization**

In [7]:
%cd /kaggle/working/PA2

for run_dir in ("outputs/dpo_hh", "outputs/dpo_uf"):
    d = Path(run_dir)
    print("=====", run_dir)
    for tag in ("best", "final"):
        print(f"  {tag}: {'OK' if (d/tag/'config.json').exists() else 'MISSING'}")
    mp = d / "metrics.jsonl"
    if mp.exists():
        rows = [json.loads(l) for l in open(mp) if l.strip()]
        val = [r for r in rows if "val_reward_acc" in r]
        print(f"  metric rows: {len(rows)} | validation rows: {len(val)}")
        for r in val:
            print(f"   step {r['step']:>4}  loss {r['val_loss']:.4f}  acc {r['val_reward_acc']:.3f}  margin {r['val_reward_margin']:.4f}")
        done = [r for r in rows if r.get("done")]
        if done: print("  summary:", done[-1])
    else:
        print("  metrics.jsonl: MISSING")

!du -sh outputs/*

zpath = "/kaggle/working/PA2_dpo_small_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in ("outputs/sft/*.json*", "outputs/dpo_hh/*.json*", "outputs/dpo_uf/*.json*", "outputs/gen/*.json"):
        for f in glob.glob(pattern):
            zf.write(f, arcname="PA2/" + f)
print("Created", zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)")


/kaggle/working
===== outputs/dpo_hh
  best: OK
  final: OK
  metric rows: 53 | validation rows: 6
   step    0  loss 0.6933  acc 0.472  margin -0.0003
   step  100  loss 0.6565  acc 0.592  margin 0.1255
   step  200  loss 0.6390  acc 0.632  margin 0.1870
   step  300  loss 0.6425  acc 0.609  margin 0.1896
   step  400  loss 0.6371  acc 0.625  margin 0.2099
   step  468  loss 0.6320  acc 0.624  margin 0.2595
  summary: {'done': True, 'best_val_reward_acc': 0.632, 'best_val_reward_margin': 0.18695904199406504}
===== outputs/dpo_uf
  best: OK
  final: OK
  metric rows: 53 | validation rows: 6
   step    0  loss 0.6929  acc 0.508  margin 0.0004
   step  100  loss 0.6017  acc 0.665  margin 0.3774
   step  200  loss 0.5829  acc 0.683  margin 0.4347
   step  300  loss 0.5765  acc 0.693  margin 0.5042
   step  400  loss 0.5681  acc 0.703  margin 0.5569
   step  468  loss 0.5650  acc 0.705  margin 0.6019
  summary: {'done': True, 'best_val_reward_acc': 0.705, 'best_val_reward_margin': 0.601878